<a href="https://colab.research.google.com/github/harshav6/DeepLearning/blob/main/InceptionV1/GoogleNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torchvision import models
from torchsummary import summary
import wandb
import os

In [3]:
# =======================
# STEP 0: Initialize wandb
# =======================
wandb.init(project="Inception-flowers", config={
    "epochs": 50,
    "batch_size": 16,
    "learning_rate": 0.001,
    "architecture": "InceptionV1",
    "pretrained": True,
    "input_size": 224
})

# Shortcut to config values
config = wandb.config

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: harshavardhanv16 (harshavardhanv16-anil) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [7]:
# =======================
# STEP 1: Data Preparation
# =======================

# Transforms for training and validation
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
}

train_dir = "/content/flowers/flowers/train"
val_dir = "/content/flowers/flowers/val"

train_dataset = datasets.ImageFolder(root=train_dir, transform=data_transforms['train'])
val_dataset = datasets.ImageFolder(root=val_dir, transform=data_transforms['val'])

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size)

In [6]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/flowers-20260820T043258Z-1-001.zip"
extract_path = "/content/flowers"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print(os.listdir(extract_path))

['flowers']


In [8]:
# ===========================
# STEP 2: Load Pretrained Model
# ===========================
from torchvision.models import GoogLeNet_Weights


# Load pretrained GoogLeNet (Inception v1)
model = models.googlenet(weights=GoogLeNet_Weights.DEFAULT)

# Replace the final FC layer to match 5 flower classes
model.fc = nn.Linear(model.fc.in_features, 5)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Unfreeze only the final classification layer
for param in model.fc.parameters():
    param.requires_grad = True

# Move model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Watch the model's weights and gradients
wandb.watch(model, log="all", log_freq=10)

Downloading: "https://download.pytorch.org/models/googlenet-1378be20.pth" to /root/.cache/torch/hub/checkpoints/googlenet-1378be20.pth


100%|██████████| 49.7M/49.7M [00:00<00:00, 143MB/s]


In [9]:
# ===================
# STEP 3: Loss & Optimizer
# ===================

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)

In [10]:
def train_model(model, criterion, optimizer, train_loader, val_loader, epochs=10):
    for epoch in range(epochs):
        model.train()
        train_correct = 0
        train_total = 0
        running_loss = 0.0

        print(f"\nEpoch {epoch + 1}/{epochs}")
        print("-" * 30)

        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            batch_correct = (preds == labels).sum().item()
            train_correct += batch_correct
            train_total += labels.size(0)

            # Print every 10 batches
            if (i + 1) % 10 == 0:
                batch_acc = batch_correct / labels.size(0)
                print(f"[Batch {i+1}/{len(train_loader)}] Loss: {loss.item():.4f}, Batch Acc: {batch_acc:.4f}")

        train_acc = train_correct / train_total
        wandb.log({"epoch": epoch + 1, "train_loss": running_loss, "train_accuracy": train_acc})
        print(f"Epoch {epoch+1} Summary - Loss: {running_loss:.4f}, Train Accuracy: {train_acc:.4f}")

        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_acc = val_correct / val_total
        wandb.log({"epoch": epoch + 1, "val_accuracy": val_acc})
        print(f"Validation Accuracy: {val_acc:.4f}")


In [11]:
# ===================
# Train the model
# ===================
train_model(model, criterion, optimizer, train_loader, val_loader, epochs=config.epochs)


Epoch 1/50
------------------------------
[Batch 10/251] Loss: 1.4928, Batch Acc: 0.3750
[Batch 20/251] Loss: 1.4978, Batch Acc: 0.3125
[Batch 30/251] Loss: 1.1526, Batch Acc: 0.7500
[Batch 40/251] Loss: 1.0080, Batch Acc: 0.8750
[Batch 50/251] Loss: 0.9843, Batch Acc: 0.6875
[Batch 60/251] Loss: 1.1771, Batch Acc: 0.4375
[Batch 70/251] Loss: 1.3949, Batch Acc: 0.3750
[Batch 80/251] Loss: 0.9448, Batch Acc: 0.7500
[Batch 90/251] Loss: 0.8909, Batch Acc: 0.7500
[Batch 100/251] Loss: 0.5826, Batch Acc: 0.9375
[Batch 110/251] Loss: 0.8383, Batch Acc: 0.6875
[Batch 120/251] Loss: 0.7710, Batch Acc: 0.8125
[Batch 130/251] Loss: 0.7509, Batch Acc: 0.7500
[Batch 140/251] Loss: 0.6096, Batch Acc: 0.8750
[Batch 150/251] Loss: 1.0847, Batch Acc: 0.5000
[Batch 160/251] Loss: 0.6953, Batch Acc: 0.8750
[Batch 170/251] Loss: 0.8487, Batch Acc: 0.7500
[Batch 180/251] Loss: 0.7968, Batch Acc: 0.8125
[Batch 190/251] Loss: 0.6309, Batch Acc: 0.8125
[Batch 200/251] Loss: 0.5667, Batch Acc: 0.8750
[Batch

KeyboardInterrupt: 